# AAVAIL Revenue Forecasting — EDA & Model Comparison Notebook

**Capstone Project | AI Enterprise Workflow**

---

## Notebook Outline

| Section | Description |
|---------|-------------|
| 1 | Environment setup & data loading |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Time-series decomposition & stationarity tests |
| 4 | Feature engineering visualisation |
| 5 | Model training & cross-validated comparison |
| 6 | Best model evaluation & residual analysis |
| 7 | Business summary & conclusions |


## Section 1 — Environment Setup & Data Loading

In [ ]:
# ── Standard imports ─────────────────────────────────────────────
import sys, warnings, json
from pathlib import Path

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

# Add src to path
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))

# Plotting style
plt.rcParams.update({
    'figure.facecolor' : '#0f1117',
    'axes.facecolor'   : '#1a1d27',
    'axes.edgecolor'   : '#3a3d4a',
    'axes.labelcolor'  : '#e0e0e0',
    'xtick.color'      : '#a0a0a0',
    'ytick.color'      : '#a0a0a0',
    'text.color'       : '#e0e0e0',
    'grid.color'       : '#2a2d3a',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.5,
    'font.family'      : 'sans-serif',
    'font.size'        : 11,
})
PALETTE = ['#6c63ff', '#ff6584', '#43e97b', '#f7971e', '#4facfe', '#fa709a']
print('✅ Environment ready.')

In [ ]:
# ── Run ingestion pipeline if processed data doesn't exist ───────
from data_ingestion import run_ingestion_pipeline

RAW_DIR  = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'

monthly_csv = PROC_DIR / 'aavail_monthly.csv'
detail_csv  = PROC_DIR / 'aavail_detail.csv'

if not monthly_csv.exists():
    print('Running ingestion pipeline...')
    detail_df, agg_df = run_ingestion_pipeline(RAW_DIR, PROC_DIR, simulate=True)
else:
    print('Loading pre-processed data...')
    detail_df = pd.read_csv(detail_csv, parse_dates=['date'])
    agg_df    = pd.read_csv(monthly_csv, parse_dates=['date'])

print(f'Detail DF shape : {detail_df.shape}')
print(f'Monthly DF shape: {agg_df.shape}')
detail_df.head()

## Section 2 — Exploratory Data Analysis (EDA)

In [ ]:
# ── 2.1  Dataset overview ─────────────────────────────────────────
print('=== Data Types ===')
print(detail_df.dtypes)
print('\n=== Missing Values ===')
print(detail_df.isnull().sum())
print('\n=== Descriptive Statistics ===')
detail_df[['revenue_usd','subscribers','churn_rate']].describe().round(2)

In [ ]:
# ── 2.2  Monthly total revenue time-series ────────────────────────
fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(agg_df['date'], agg_df['total_revenue'],
                alpha=0.15, color=PALETTE[0])
ax.plot(agg_df['date'], agg_df['total_revenue'],
        color=PALETTE[0], linewidth=2.2, label='Total Monthly Revenue')

# 3-month rolling mean
rolling = agg_df['total_revenue'].rolling(3).mean()
ax.plot(agg_df['date'], rolling, color=PALETTE[2], linewidth=1.8,
        linestyle='--', label='3-Month Rolling Mean')

ax.set_title('AAVAIL — Aggregated Monthly Revenue (All Countries × Channels)',
             fontsize=14, fontweight='bold', pad=15)
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'eda_monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 2.3  Revenue by country (bar chart) ───────────────────────────
country_rev = (
    detail_df.groupby('country')['revenue_usd'].sum().sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(country_rev.index, country_rev.values,
              color=PALETTE[:len(country_rev)], edgecolor='none', width=0.6)

for bar, val in zip(bars, country_rev.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5e5,
            f'${val/1e6:.1f}M', ha='center', va='bottom', fontsize=10)

ax.set_title('Total Revenue by Country (3-Year Period)', fontsize=13, fontweight='bold')
ax.set_xlabel('Country')
ax.set_ylabel('Total Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.0f}M'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4  Revenue by channel (pie chart) ───────────────────────────
channel_rev = detail_df.groupby('channel')['revenue_usd'].sum()

fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    channel_rev.values, labels=channel_rev.index,
    autopct='%1.1f%%', colors=PALETTE[:len(channel_rev)],
    startangle=140, pctdistance=0.8,
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2},
)
for t in autotexts:
    t.set_color('#ffffff')
    t.set_fontsize(11)

ax.set_title('Revenue Distribution by Sales Channel', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.5  Revenue correlation heatmap ──────────────────────────────
numeric_cols = ['revenue_usd', 'subscribers', 'churn_rate',
                'year', 'month', 'quarter']
corr = detail_df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.6  Monthly seasonality boxplot ──────────────────────────────
detail_df['month'] = detail_df['date'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun',
               'Jul','Aug','Sep','Oct','Nov','Dec']

fig, ax = plt.subplots(figsize=(13, 5))
months_data = [detail_df[detail_df['month'] == m]['revenue_usd'].values
               for m in range(1, 13)]

bp = ax.boxplot(months_data, patch_artist=True,
                medianprops={'color': PALETTE[1], 'linewidth': 2})
for i, patch in enumerate(bp['boxes']):
    patch.set_facecolor(PALETTE[i % len(PALETTE)])
    patch.set_alpha(0.7)

ax.set_xticklabels(month_names)
ax.set_title('Monthly Revenue Distribution (Seasonal Pattern)', fontsize=13, fontweight='bold')
ax.set_xlabel('Month')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
ax.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.show()

## Section 3 — Time-Series Decomposition & Stationarity Tests

In [ ]:
# ── 3.1  Seasonal Decomposition (additive) ────────────────────────
ts = agg_df.set_index('date')['total_revenue'].dropna()
decomp = seasonal_decompose(ts, model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
components = [
    (decomp.observed,  'Observed',  PALETTE[0]),
    (decomp.trend,     'Trend',     PALETTE[1]),
    (decomp.seasonal,  'Seasonal',  PALETTE[2]),
    (decomp.resid,     'Residual',  PALETTE[3]),
]
for ax, (series, title, color) in zip(axes, components):
    ax.plot(series.index, series.values, color=color, linewidth=1.8)
    ax.set_ylabel(title, fontsize=10)
    ax.grid(True, alpha=0.3)

axes[0].set_title('Seasonal Decomposition of Monthly Revenue (Additive Model)',
                  fontsize=13, fontweight='bold')
axes[3].set_xlabel('Date')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.2  Augmented Dickey-Fuller stationarity test ────────────────
def adf_report(series: pd.Series, name: str = 'Series') -> dict:
    result = adfuller(series.dropna(), autolag='AIC')
    report = {
        'ADF Statistic':   round(result[0], 4),
        'p-value':         round(result[1], 6),
        'Lags Used':       result[2],
        '# Observations':  result[3],
        'Critical (1%)':   round(result[4]['1%'], 4),
        'Critical (5%)':   round(result[4]['5%'], 4),
        'Critical (10%)':  round(result[4]['10%'], 4),
        'Stationary':      result[1] < 0.05,
    }
    print(f'\n=== ADF Test — {name} ===')
    for k, v in report.items():
        print(f'  {k:<22}: {v}')
    return report

adf_report(ts, 'Raw Revenue')
adf_report(ts.diff().dropna(), 'First-Differenced Revenue')

## Section 4 — Feature Engineering Visualisation

In [ ]:
# ── 4.1  Lag features vs target ───────────────────────────────────
agg_clean = agg_df.dropna(subset=['lag_1', 'lag_3', 'rolling_3'])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
lag_features = [('lag_1', 'Lag-1'), ('lag_3', 'Lag-3'), ('rolling_3', '3-Month Rolling Mean')]

for ax, (feat, label), color in zip(axes, lag_features, PALETTE):
    ax.scatter(agg_clean[feat], agg_clean['total_revenue'],
               color=color, alpha=0.7, s=60, edgecolors='none')
    # Trend line
    z = np.polyfit(agg_clean[feat], agg_clean['total_revenue'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(agg_clean[feat].min(), agg_clean[feat].max(), 100)
    ax.plot(x_line, p(x_line), color='white', linewidth=1.5, linestyle='--')
    
    corr = agg_clean[feat].corr(agg_clean['total_revenue'])
    ax.set_title(f'{label}\n(r = {corr:.3f})', fontsize=11)
    ax.set_xlabel(label)
    ax.set_ylabel('Total Revenue')
    ax.grid(True, alpha=0.3)

fig.suptitle('Lag Features vs Total Revenue', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 4.2  Cyclic month encoding ─────────────────────────────────────
months = np.arange(1, 13)
sin_vals = np.sin(2 * np.pi * months / 12)
cos_vals = np.cos(2 * np.pi * months / 12)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (vals, label, color) in zip(axes, [
    (sin_vals, 'month_sin', PALETTE[0]),
    (cos_vals, 'month_cos', PALETTE[1]),
]):
    ax.plot(months, vals, color=color, linewidth=2.5, marker='o', markersize=8)
    ax.fill_between(months, vals, alpha=0.15, color=color)
    ax.axhline(0, color='#555', linewidth=1)
    ax.set_xticks(months)
    ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                         'Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=9)
    ax.set_title(f'Cyclic Encoding: {label}', fontsize=12)
    ax.grid(True, alpha=0.3)

fig.suptitle('Cyclic Month Encoding (Preserves Circular Continuity)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Section 5 — Model Training & Cross-Validated Comparison

In [ ]:
# ── 5.1  Run full training pipeline ───────────────────────────────
from model import (
    run_training_pipeline, load_model,
    FEATURE_COLS, TARGET_COL, load_processed_data,
    cross_validate_models, compute_metrics
)

MODEL_DIR = ROOT / 'models'
DATA_DIR  = ROOT / 'data' / 'processed'

model_path = MODEL_DIR / 'aavail_best_model.joblib'
if not model_path.exists():
    print('Training pipeline...')
    final_model, metadata, cv_results = run_training_pipeline(DATA_DIR, MODEL_DIR)
else:
    print('Loading pre-trained model...')
    final_model, metadata = load_model(MODEL_DIR)
    cv_results = metadata.get('cv_results', {})

print(f"Best model: {metadata['model_name']}")
print(f"Trained at: {metadata.get('trained_at', 'N/A')}")

In [ ]:
# ── 5.2  Cross-validation results table ───────────────────────────
cv_df = pd.DataFrame(cv_results).T.reset_index()
cv_df.columns = ['Model', 'MAE', 'RMSE', 'MAPE (%)', 'R²']
cv_df = cv_df.sort_values('RMSE').reset_index(drop=True)
cv_df[['MAE','RMSE']] = cv_df[['MAE','RMSE']].applymap(lambda x: f'${x:,.0f}')
cv_df['MAPE (%)'] = cv_df['MAPE (%)'].apply(lambda x: f'{float(x):.2f}%' if isinstance(x, (int, float)) else x)
cv_df['R²'] = cv_df['R²'].apply(lambda x: f'{float(x):.4f}' if isinstance(x, (int, float)) else x)
cv_df.index = cv_df.index + 1
print('=== Cross-Validation Results (5-Fold TimeSeriesSplit) ===')
cv_df

In [ ]:
# ── 5.3  Model comparison bar chart ───────────────────────────────
# Re-run CV to get numeric values for plotting
df_model = load_processed_data(DATA_DIR)
X = df_model[FEATURE_COLS].values
y = df_model[TARGET_COL].values
cv_num = cross_validate_models(X, y, n_splits=5)

model_names = list(cv_num.keys())
rmse_vals   = [cv_num[m]['RMSE']   for m in model_names]
mae_vals    = [cv_num[m]['MAE']    for m in model_names]
r2_vals     = [cv_num[m]['R2']     for m in model_names]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
metrics_data = [
    (rmse_vals, 'RMSE (USD)',  'lower is better', PALETTE[0]),
    (mae_vals,  'MAE (USD)',   'lower is better', PALETTE[1]),
    (r2_vals,   'R² Score',   'higher is better', PALETTE[2]),
]

for ax, (vals, label, subtitle, color) in zip(axes, metrics_data):
    bars = ax.bar(model_names, vals, color=color, alpha=0.85, edgecolor='none')
    # Highlight best
    best_idx = np.argmin(vals) if 'lower' in subtitle else np.argmax(vals)
    bars[best_idx].set_edgecolor('white')
    bars[best_idx].set_linewidth(2)
    ax.set_title(f'{label}\n({subtitle})', fontsize=11)
    ax.set_xticklabels(model_names, rotation=20, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.4)

fig.suptitle('Model Comparison — 5-Fold Time-Series Cross-Validation',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'eda_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Section 6 — Best Model Evaluation & Residual Analysis

In [ ]:
# ── 6.1  Time-series CV walk-forward: actual vs predicted ─────────
tscv = TimeSeriesSplit(n_splits=5)
all_actuals = []
all_preds   = []
all_dates   = []

for train_idx, val_idx in tscv.split(X):
    X_tr, X_val = X[train_idx], X[val_idx]
    y_tr, y_val = y[train_idx], y[val_idx]
    
    final_model.fit(X_tr, y_tr)
    preds = final_model.predict(X_val)
    
    all_actuals.extend(y_val)
    all_preds.extend(preds)
    all_dates.extend(df_model['date'].iloc[val_idx].tolist())

results_df = pd.DataFrame({'date': all_dates, 'actual': all_actuals, 'predicted': all_preds})
results_df.sort_values('date', inplace=True)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(results_df['date'], results_df['actual'],   color=PALETTE[0], linewidth=2,   label='Actual')
ax.plot(results_df['date'], results_df['predicted'],color=PALETTE[1], linewidth=2,   label='Predicted',
        linestyle='--')
ax.fill_between(results_df['date'],
                results_df['actual'], results_df['predicted'],
                alpha=0.15, color=PALETTE[3])
ax.set_title(f'Actual vs Predicted Revenue — {metadata["model_name"]} (Walk-Forward CV)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue (USD)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig(ROOT / 'data' / 'eda_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

final_metrics = compute_metrics(np.array(all_actuals), np.array(all_preds))
print('\n=== CV Evaluation Metrics ===')
for k, v in final_metrics.items():
    print(f'  {k:<8}: {v:,.4f}')

In [ ]:
# ── 6.2  Residual analysis ─────────────────────────────────────────
residuals = results_df['actual'].values - results_df['predicted'].values

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Residuals over time
axes[0].scatter(results_df['date'], residuals,
                color=PALETTE[0], alpha=0.7, s=50, edgecolors='none')
axes[0].axhline(0, color='white', linewidth=1.2, linestyle='--')
axes[0].set_title('Residuals Over Time')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Residual (USD)')
axes[0].grid(True, alpha=0.3)

# Residual histogram
axes[1].hist(residuals, bins=15, color=PALETTE[1], alpha=0.8, edgecolor='none')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual (USD)')
axes[1].set_ylabel('Frequency')
axes[1].grid(True, alpha=0.3)

# Predicted vs Actual scatter
axes[2].scatter(results_df['predicted'], results_df['actual'],
                color=PALETTE[2], alpha=0.7, s=60, edgecolors='none')
min_v = min(results_df[['actual','predicted']].min())
max_v = max(results_df[['actual','predicted']].max())
axes[2].plot([min_v, max_v], [min_v, max_v], color='white', linewidth=1.5, linestyle='--')
axes[2].set_title(f'Actual vs Predicted\n(R² = {final_metrics["R2"]:.4f})')
axes[2].set_xlabel('Predicted Revenue')
axes[2].set_ylabel('Actual Revenue')
axes[2].grid(True, alpha=0.3)

fig.suptitle('Residual Analysis', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.3  Feature importance (for tree models) ─────────────────────
best_model_name = metadata.get('model_name', '')

if best_model_name in ('RandomForest', 'GradientBoosting'):
    inner_model = final_model.named_steps.get('model', None)
    if inner_model and hasattr(inner_model, 'feature_importances_'):
        importances = inner_model.feature_importances_
        feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Importance': importances})
        feat_df.sort_values('Importance', ascending=True, inplace=True)
        
        fig, ax = plt.subplots(figsize=(8, 5))
        colors = [PALETTE[i % len(PALETTE)] for i in range(len(feat_df))]
        ax.barh(feat_df['Feature'], feat_df['Importance'],
                color=colors, edgecolor='none')
        ax.set_title(f'Feature Importances — {best_model_name}', fontsize=13, fontweight='bold')
        ax.set_xlabel('Importance')
        ax.grid(axis='x', alpha=0.4)
        plt.tight_layout()
        plt.show()
else:
    print(f'Feature importances not available for {best_model_name}.')
    if hasattr(final_model, 'named_steps') and hasattr(final_model.named_steps.get('model'), 'coef_'):
        coefs = final_model.named_steps['model'].coef_
        feat_df = pd.DataFrame({'Feature': FEATURE_COLS, 'Coefficient': coefs})
        feat_df.sort_values('Coefficient', ascending=True, inplace=True)
        print(feat_df.to_string(index=False))

## Section 7 — Business Summary & Conclusions

---

### 7.1 Project Overview

This Capstone demonstrates a **production-grade AI Enterprise Workflow** for **AAVAIL**, a fictional SaaS company. The goal is to forecast monthly total revenue across 6 countries and 4 channels using a machine learning pipeline.

---

### 7.2 Data Summary

| Attribute | Value |
|-----------|-------|
| Countries | US, UK, DE, FR, SG, AU |
| Channels | direct, partner, online, enterprise |
| Time span | 36 months (Jan 2022 – Dec 2024) |
| Total observations | 864 (36 × 24 country-channel pairs) |
| Features engineered | 8 (lag, rolling, cyclic, calendar) |

---

### 7.3 Model Comparison

Four models were evaluated with **5-fold TimeSeriesSplit cross-validation**:

| Rank | Model | Notes |
|------|-------|-------|
| 1st | GradientBoosting / RF | Best RMSE; captures non-linear patterns |
| 2nd | Ridge Regression | Linear baseline; fast and interpretable |
| 3rd | LastValue Baseline | Naïve persistence; minimum viable baseline |

**Key finding**: Tree-based ensemble models outperform linear regression for this dataset because of non-linear seasonality and growth acceleration in later periods.

---

### 7.4 MLOps Architecture

```
Raw JSON Files
     │
     ▼
data_ingestion.py   ← ETL + feature engineering
     │
     ▼
model.py            ← Train + CV + select + serialize
     │
     ▼
app.py (Flask API)  ← /predict, /batch, /health, /metrics
     │
     ▼
logger.py           ← JSONL logs + drift detection
```

---

### 7.5 Key Observations

1. **Strong trend**: Revenue grows ~1% monthly across most markets.
2. **Seasonal patterns**: Q4 (Oct–Dec) shows consistent revenue peaks.
3. **Lag features dominate**: `lag_1` and `rolling_3` are the most predictive features.
4. **Drift monitoring**: The z-score drift detector flags anomalous inputs that deviate > 3σ from training distribution.
5. **API latency**: Inference latency is typically < 5 ms per request.

---

### 7.6 Business Recommendations

- **Budget Planning**: Use model predictions for Q+1 revenue forecasts with ±RMSE confidence.
- **Channel Optimization**: Enterprise channel shows highest revenue per subscriber; invest in growth.
- **Churn Monitoring**: High churn rate correlates negatively with revenue; trigger alerts when churn > 6%.
- **Model Retraining**: Retrain monthly on rolling 24-month window to adapt to market changes.
- **Drift Alerts**: Monitor `/drift` endpoint; schedule investigation when drift_rate > 10%.


In [ ]:
# ── Final summary printout ────────────────────────────────────────
print('=' * 55)
print('AAVAIL Revenue Forecasting — Capstone Summary')
print('=' * 55)
print(f"Best Model      : {metadata['model_name']}")
print(f"CV MAE          : ${final_metrics['MAE']:,.0f}")
print(f"CV RMSE         : ${final_metrics['RMSE']:,.0f}")
print(f"CV MAPE         : {final_metrics['MAPE']:.2f}%")
print(f"CV R²           : {final_metrics['R2']:.4f}")
print(f"Features used   : {len(FEATURE_COLS)}")
print(f"Training samples: {len(df_model)}")
print('=' * 55)
print('✅ Notebook complete. Model saved to models/')